# How Do People Feel About Facebook? Sentiment Analysis of Tweets
### <span style="color: blue;"> A hands-on introduction to computational sentiment analysis with VADER
##### October 2026, Sheryl Winston Smith

**<span style="color: green;">Business question</span>** Founders and product managers want to know how customers *feel*: about a launch, a feature, a competitor. Social media is full of opinions, but there are far too many posts to read. Can a computer read them for us, and can we trust what it says?

**<span style="color: green;">What is this?</span>** 176 unique tweets that mention Facebook, all posted in a 30-minute window on 1 October 2015. That was the day Facebook announced *profile videos* and the week the controversial startup *Peeple* (a "Yelp for people") launched. You already coded a few by hand on the handout. Now we see how an algorithm does it.

**<span style="color: green;">What will you learn?</span>**

| Part | Topic | Key idea |
|---|---|---|
| 1 | Tweets as data | Cleaning choices (duplicates, line breaks, IDs) change your results |
| 2 | How VADER works | A dictionary of rated words plus a few grammar rules |
| 3 | Scores to labels | A cut-off turns a number into "positive / neutral / negative" |
| 4 | Experiments | Edit words, emoji, CAPS and !!! and watch the score move |
| 5 | You vs. VADER | Validity: does the machine agree with careful humans? |
| 6 | Traps in real data | Sarcasm, other languages, spam, bots, off-target sentiment |
| 7 | Beyond the dictionary (optional) | How transformer models differ |

<span style="color: blue;">Look for cells marked **🔧 Try it**. Those are where you change something and re-run.

> <span style="color: green;"> **Runs offline in seconds.** Parts 1 to 6 need only `pandas`, `matplotlib` and `vaderSentiment`. Part 7 is optional and is switched off by default.

## 0. Setup

Keep the `data/` folder next to this notebook. Required packages: `pandas`, `matplotlib`, `vaderSentiment`. In your environment (terminal) run `pip install pandas matplotlib vaderSentiment`.

In [ ]:
import html, re, string, textwrap
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DATA = Path("data")          # folder that sits next to this notebook
pd.set_option("display.max_colwidth", 110)

# IDs are long numbers: read them as text, or pandas may round them
raw      = pd.read_csv(DATA / "tweets_raw.csv",   dtype={"tweet_id": str})
tweets   = pd.read_csv(DATA / "tweets_clean.csv", dtype={"tweet_id": str})
excerpts = pd.read_csv(DATA / "excerpts.csv",     dtype={"tweet_id": str})

vader = SentimentIntensityAnalyzer()

def show(text, width=100):
    print(textwrap.fill(text, width))

print(f"{len(raw)} tweets in the raw file, {len(tweets)} after removing duplicates, {len(excerpts)} teaching excerpts")

### Who is who: the teaching excerpts

| ID | Tweet starts with... | What it tests |
|---|---|---|
| **W1** | "I love that Facebook lets us see our memories..." | Worked example: clear positive |
| **W2** | "Today, 3 years ago, the Angels unfortunately..." | Worked example: sad tweet, but not *about* Facebook |
| **H1** | "Deleted Facebook app from phone. It's amazing..." | Positive words, negative stance |
| **H2** | "Facebook's new layout changes sound so stupid..." | Clear case |
| **H3** | "Deactivating Facebook is still one of the best choices..." | Positive words, negative stance |
| **H4** | "Esse facebook ta me irritando" | Not in English *(trap)* |
| **H5** | "300 Facebook fan page likes for $3" | Spam *(trap)* |
| **H6** | "Not sure I like the sound of Twitter's plan..." | Mostly about something else |
| **H7** | "I cannot believe Im activelly enjoying a Facebook game" | Clear case |
| **H8** | "Do you still use Facebook ? #RNAMERCY" | Posted 13 times within one second *(trap)* |
| **E1** | "Oh hey, facebook! You turned yourself back on..." | Clear case |
| **E2** | "Facebook app keeps on closing. WHYYYY?" | Anger with no "emotion words" |
| **E3** | "Facebook, I didn't ask you to autoplay every video..." | Clear case |
| **E4** | "Facebook has gotten worse over the years..." | Clear case |
| **E5** | "The new @Facebook feature coming is profile videos..." | Polite skepticism |
| **E6** | "I prefer facebook but oh well I like tweeting👌👌👌" | Emoji |
| **E7** | "Only 1 % Girls become / Wife of their lovers..." | Multi-line joke with emoji |
| **E8** | "Jazz free facebook / code / Dail *442*8*1#..." | Spam, multi-line, mixed language *(trap)* |
| **E9** | "Successful Farming study shows 34% farmers use Facebook..." | A name that looks like praise *(trap)* |
| **E10** | "Mas paspas pang youtube kaysa facebook..." | Not in English *(trap)* |
| **E11** | "Can't even use tinder at school cause they have Facebook blocked 😭" | Sentiment about something else |
| **E12** | "#Peeple to increase hatred... I thought Facebook had all of that covered already" | Sarcasm |

H1 to H8 are the tweets from the handout. W1 and W2 are its worked examples.

## 1. Tweets as data

Before any model runs, someone makes cleaning decisions. Each one changes the results. Here are three that mattered for this file.

In [ ]:
# (a) Tweets can contain line breaks. A careless reader (e.g. splitting the file on new lines)
#     would chop them into fragments. pandas respects the quotes around each tweet.
lines_in_file = (DATA / "tweets_raw.csv").read_text(encoding="utf-8").count("\n")
print(f"Lines in the file: {lines_in_file}, but tweets: {len(raw)}")
print(f"Tweets that span several lines: {raw.tweet_text.str.contains(chr(10)).sum()}\n")
print(raw.loc[raw.tweet_text.str.contains(chr(10)), "tweet_text"].iloc[0])

In [ ]:
# (b) Tweet IDs are 18-digit numbers. Read as numbers, they lose their last digits.
as_number = pd.read_csv(DATA / "tweets_raw.csv").tweet_id.astype(float)
print("Read as text:  ", raw.tweet_id.iloc[0])
print("Read as number:", f"{as_number.iloc[0]:.0f}")

In [ ]:
# (c) Duplicates: the same text posted again and again
dupes = raw.tweet_text.value_counts()
dupes[dupes > 1]

In [ ]:
# When were they posted?
rna = raw[raw.tweet_text.isin(dupes[dupes > 1].index)]
print(f"{len(rna)} duplicate tweets")
print(rna.groupby("tweet_text").created_at.agg(["count", "min", "max"]))
rna[["tweet_id", "created_at", "tweet_text"]].head(6)

<span style="color: blue;">**Go deeper:** Each question was posted 13 times within about one second, and the two bursts came 18 seconds apart. (We removed the user IDs from this file, but the original data showed the same 13 accounts behind both bursts.) These are not 26 people with an opinion. They look like a coordinated campaign (bots, or one person running many accounts). If we left them in, 13% of our "sample" would be two neutral questions.

<span style="color: blue;">**Our cleaning choices:** keep one copy of each exact text (`n_copies` records how many there were), turn HTML codes like `&amp;` back into `&`, and add hand-made flags for spam and non-English tweets. The cleaned file is `tweets_clean.csv`.

In [ ]:
tweets[["text", "n_copies", "multi_line", "is_spam", "language", "compound"]].sample(8, random_state=1)

## 2. How VADER works

### <span style="color: green;">Q: What is VADER, and is it "AI"?

<span style="color: blue;">**A:** VADER (Valence Aware Dictionary and sEntiment Reasoner; Hutto & Gilbert, 2014) is a **lexicon- and rule-based** model built for social media. It is *not* a transformer or a large language model, and it does not learn from data when you run it. It has two parts:

| Part | What it is | Example |
|---|---|---|
| **Lexicon** | About 7,500 words, slang terms, emoticons and emoji, each rated by human raters from −4 (very negative) to +4 (very positive) | "love" = +3.2, "stupid" = −2.4, ":)" = +2.0 |
| **Rules** | A handful of hand-written grammar heuristics | Negation ("not good"), intensifiers ("very"), CAPS, "!!!", and "but" (what comes after "but" counts more) |

<span style="color: blue;">VADER adds up the ratings of the words it finds, adjusts them with the rules, and squashes the total into a **compound** score from −1 to +1. It also reports what share of the text is positive, neutral and negative.

**Why use it?** It is free, instant, fully transparent (you can look up every number), and needs no training data. **Why be careful?** It only knows the words in its dictionary. It cannot tell *what* a feeling is about, and it does not understand sarcasm.

### <span style="color: green;">Q: Then what is a transformer model?

<span style="color: blue;">**A:** A transformer (the architecture behind BERT, RoBERTa and modern LLMs) reads the *whole sentence at once* and represents each word in its context, using patterns learned from millions of examples. A sentiment transformer is trained on labelled texts, for example tweets labelled positive / neutral / negative. It can catch some things VADER misses, but it is a black box and needs much more computing power. Part 7 shows how to compare the two.

In [ ]:
# Look inside the dictionary. 🔧 Try it: add your own words
for word in ["love", "great", "amazing", "stupid", "hate", "lmao", "lol", "wtf", "facebook", "fix", "crashing"]:
    print(f"{word:>10}: {vader.lexicon.get(word, '(not in the lexicon)')}")

In [ ]:
# Emoji are translated into words first, and those words are looked up
for e in ["😂", "😭", "👌", "😇", "👎"]:
    desc = vader.emojis.get(e, "?")
    print(e, "→", f"{desc!r:<35}", "→ compound", vader.polarity_scores(e)["compound"])

In [ ]:
def explain(text):
    # Show which words VADER found in its lexicon. The rules then adjust these values.
    found = []
    for token in text.split():
        word = token.strip(string.punctuation).lower()
        if word in vader.lexicon:
            found.append((token, vader.lexicon[word]))
        for ch in token:
            if ch in vader.emojis:
                for w in vader.emojis[ch].split():
                    if w in vader.lexicon:
                        found.append((f"{ch} ({w})", vader.lexicon[w]))
    print(text.replace("\n", " / "))
    print("  words VADER rated:", found or "none")
    print("  scores:", vader.polarity_scores(text), "\n")

# 🔧 Try it: change the excerpt ID (W1, W2, H1 ... H8, E1 ... E12)
for eid in ["W1", "H1", "E9"]:
    explain(excerpts.set_index("excerpt_id").loc[eid, "text"])

<span style="color: blue;">**Go deeper:** In H1 VADER finds "amazing" and scores the tweet as positive. A human sees someone who deleted Facebook. In E9 the only rated words are "Successful" (the name of a farming magazine) and "Congrats". VADER is doing exactly what it was designed to do. The problem is that its dictionary has no idea *who or what* the words are about.

## 3. Scoring the excerpts, live

The data file already has VADER columns (`negative`, `neutral`, `positive`, `compound`). Let's re-compute them ourselves and check that we get the same numbers. **Reproducibility** is the first test of any measurement.

In [ ]:
live = pd.DataFrame([vader.polarity_scores(t) for t in raw.tweet_text])
same = (live["compound"] - raw["compound"]).abs() < 0.0001
print(f"Live compound score matches the stored column for {same.sum()} of {len(raw)} tweets")

In [ ]:
excerpts["vader_live"] = excerpts.text.apply(lambda t: vader.polarity_scores(t)["compound"])
excerpts[["excerpt_id", "text", "negative", "neutral", "positive", "vader_live"]]

<span style="color: blue;">**Note:** the match is exact here because the stored columns were produced with the same VADER version (3.3.2). A different version, or cleaning the text first (for example removing emoji), would give different numbers. Always record which version and which text you scored.

## 4. From scores to labels

The compound score is a number. To get "positive / neutral / negative" you need a **cut-off**. The VADER authors suggest ±0.05: at least +0.05 is positive, at most −0.05 is negative, anything in between is neutral.

In [ ]:
# 🔧 Try it: change the threshold (try 0.05, 0.3, 0.5), then re-run this cell and the next one
THRESHOLD = 0.05

def to_label(compound, t=None):
    t = THRESHOLD if t is None else t
    if compound >= t:
        return "positive"
    if compound <= -t:
        return "negative"
    return "neutral"

tweets["vader_label"] = tweets.compound.apply(to_label)
excerpts["vader_label"] = excerpts.vader_live.apply(to_label)
tweets.vader_label.value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(tweets.compound, bins=40, color="steelblue")
for x in (-THRESHOLD, THRESHOLD):
    ax.axvline(x, color="black", ls="--", lw=1)
ax.set_xlabel("VADER compound score"); ax.set_ylabel("tweets")
ax.set_title(f"176 unique tweets about Facebook (dashed lines: ±{THRESHOLD})")
plt.tight_layout(); plt.show()

In [ ]:
# How much does the cut-off matter? Share of each label for several thresholds
pd.DataFrame({t: tweets.compound.apply(to_label, t=t).value_counts(normalize=True).round(2)
              for t in [0.05, 0.2, 0.4, 0.6]}).rename_axis("threshold →", axis=1)

<span style="color: blue;">**Go deeper:** Notice the tall bar at exactly 0. Those tweets contain *no* word from the lexicon. "Neutral" in VADER often means "I found nothing to rate", not "this person feels neutral". E2 ("keeps on closing. WHYYYY?") and both non-English excerpts sit at 0.

<span style="color: blue;">**🔧 Try it:** Suppose you report "X% of tweets about Facebook are positive" to an investor. How much can you move X just by choosing the threshold? Who should choose it, and when?

## 5. Experiments: change the text, watch the score

Because VADER is a set of rules, you can test each rule directly. Change one thing at a time.

In [ ]:
def compare(*versions):
    return pd.DataFrame([{"text": v, **vader.polarity_scores(v)} for v in versions])

compare(
    "Facebook is great",
    "Facebook is GREAT",            # caps
    "Facebook is great!!!",         # exclamation marks
    "Facebook is very great",       # intensifier
    "Facebook is kind of great",    # hedge
    "Facebook is not great",        # negation
    "Facebook is great but slow",   # "but"
    "Facebook is great 😂",          # emoji
)

In [ ]:
# 🔧 Try it: edit a real tweet. Remove the emoji, change a word, add "not", add CAPS or !!!
compare(
    "I prefer facebook but oh well I like tweeting👌👌👌",
    "I prefer facebook but oh well I like tweeting",
    "Deleted Facebook app from phone. It's amazing how much battery I have left at end of day.",
    "Deleted Facebook app from phone. Good riddance.",
    "Facebook app keeps on closing. WHYYYY?",
    "Facebook app keeps on closing. So annoying!",
)

<span style="color: blue;">**Go deeper:** Removing three 👌 emoji moves E6 from 0.99 to 0.71. Rewriting H1 as "Good riddance" barely helps: "good" is positive in the lexicon, so VADER still reads approval. The second E2 version gets a negative score only because "annoying" is in the dictionary.

<span style="color: blue;">**🔧 Try it:** Write one tweet that a human would call clearly *negative* about Facebook but that VADER scores as positive. What does this tell you about using VADER on your own customer feedback?

## 6. You vs. VADER

<span style="color: blue;">The real test of any method is agreement with careful human judgment. Enter the labels you wrote on the handout for H1 to H8 (`"positive"`, `"neutral"` or `"negative"`), then run the next cells.

In [ ]:
# 🔧 Try it: fill in your own labels from the handout
my_labels = {
    "H1": "",
    "H2": "",
    "H3": "",
    "H4": "",
    "H5": "",
    "H6": "",
    "H7": "",
    "H8": "",
}

In [ ]:
ex = excerpts.set_index("excerpt_id")
you = pd.DataFrame([{
    "excerpt": eid,
    "tweet": ex.loc[eid, "text"][:60],
    "you": lab or "(blank)",
    "VADER": ex.loc[eid, "vader_label"],
    "compound": ex.loc[eid, "vader_live"],
    "agree": lab == ex.loc[eid, "vader_label"],
} for eid, lab in my_labels.items()])
print(f"You and VADER agree on {you.agree.sum()} of {sum(map(bool, my_labels.values()))} labelled tweets")
you

## <span style="color: green;">Answer key (suggested)

<span style="color: blue;">Fill in your own labels **before** reading this. The question is sentiment **toward Facebook**, not the overall mood of the tweet. Reasonable coders disagree on the borderline cases, and those disagreements are part of the lesson.

| ID | Suggested label | VADER | Why |
|---|---|---|---|
| **H1** | negative | positive | Deleting the app rejects Facebook; "amazing" praises the battery life. |
| **H2** | negative | negative | "so stupid" is aimed at Facebook's layout changes. |
| **H3** | negative | positive | "best choices" praises *leaving*; Facebook is the source of "negativity". |
| **H4** | negative | neutral | Portuguese: "This Facebook is irritating me." VADER knows no Portuguese. |
| **H5** | neutral (or exclude) | positive | An ad selling fake likes. No opinion about Facebook at all. |
| **H6** | negative | negative | Mainly about Twitter, but "turn into Facebook and that's not good" judges Facebook. *Debate:* is a comparison an opinion? |
| **H7** | positive | positive | Enjoying a Facebook game. |
| **H8** | neutral | neutral | A question with no opinion, and one of 13 identical copies. |

The code cell below holds the key for all 22 excerpts.

In [ ]:
key = excerpts.set_index("excerpt_id")[["suggested_label", "vader_label", "vader_live", "trap", "why"]]
key["agree"] = key.suggested_label == key.vader_label
print(f"VADER agrees with the suggested key on {key.agree.sum()} of {len(key)} excerpts "
      f"({key.agree.mean():.0%}). With 3 labels, guessing at random would give about 33%.\n")
print(pd.crosstab(key.suggested_label, key.vader_label, rownames=["key (human)"], colnames=["VADER"]))
key

In [ ]:
# How did you do against the key?
filled = {k: v for k, v in my_labels.items() if v}
if filled:
    hits = sum(v == key.loc[k, "suggested_label"] for k, v in filled.items())
    print(f"You agree with the key on {hits} of {len(filled)} tweets")
    print(f"VADER agrees with the key on {key.loc[list(filled), 'agree'].sum()} of the same {len(filled)}")
else:
    print("Fill in my_labels above first.")

<span style="color: blue;">**Go deeper:** Look at where VADER disagrees with the key. Almost every miss falls into one of three groups: (1) the feeling is real but aimed at something else (W2, H1, H3, E11), or a positive word is part of a name (E9); (2) VADER found no words it knows (H4, E2, E10); (3) the text is not an opinion at all (H5, E7, E8). E5 is a fourth kind: polite skepticism that uses the word "hope". We picked hard cases on purpose, so agreement on these 22 excerpts probably understates how VADER does on typical tweets. In your own work, measure agreement on a **random** sample.

## 7. Traps in real data

Here is how much each trap affects the full dataset, not just our excerpts.

In [ ]:
def share_positive(df):
    return (df.compound.apply(to_label) == "positive").mean()

def mean_score(df):
    return df.compound.mean()

versions = {
    "raw file (200, with duplicates)": raw,
    "duplicates removed (176)":        tweets,
    "...and spam removed":             tweets[~tweets.is_spam],
    "...and English only":             tweets[~tweets.is_spam & (tweets.language == "English")],
}
pd.DataFrame([{"version": k, "tweets": len(v), "share positive": round(share_positive(v), 2),
               "mean compound": round(mean_score(v), 3)} for k, v in versions.items()])

<span style="color: blue;">**Go deeper:**
* **Duplicates and bots** inflate whatever they say. Here they were neutral, so they pulled the average toward 0.
* **Spam** reads as positive ("free", "likes", "100% working"). Ads selling Facebook likes are not happy customers.
* **Non-English tweets** almost always score 0 because the lexicon is English. They are not neutral; they are unmeasured.
* **Multi-line tweets** are fine *if* the file is read correctly. Read naively, one tweet becomes several fragments with their own scores.
* **Sarcasm** ("I thought Facebook had all of that covered already") flips meaning without changing words.
* **Off-target sentiment** is the biggest trap: "mentions Facebook" is not the same as "is about Facebook".

In [ ]:
# 🔧 Try it: a mini market-research question. Facebook announced profile videos that day.
# What does VADER say about tweets that mention "video"? Read them: do you agree?
# Then try other keywords: "peeple", "down", "messenger", "deleted|deactivat"
KEYWORD = "video"
subset = tweets[tweets.text.str.contains(KEYWORD, case=False)]
print(f"{len(subset)} tweets mention {KEYWORD!r}; mean compound {subset.compound.mean():+.2f}\n")
for _, r in subset.iterrows():
    print(f"{r.compound:+.2f}  {r.text.replace(chr(10), ' / ')[:110]}")

<span style="color: blue;">**Go deeper:** Several "video" tweets are about autoplay and video-heavy feeds, not profile videos, and the average hides a wide spread. With a dozen tweets from a 30-minute window, would you tell a product team how users feel about the launch? What sample would you need?

<span style="color: blue;">**Peeple** was a startup, and its launch week produced a wave of tweets. Most mention Facebook only as a comparison. If you were Peeple's founders, which tweets would you want, and how would you collect *those* rather than tweets that happen to mention a bigger brand?

## 8. Beyond the dictionary: transformer models (optional)

### <span style="color: green;">Q: What changes with a transformer?

<span style="color: blue;">**A:** A transformer-based classifier such as `cardiffnlp/twitter-roberta-base-sentiment-latest` was trained on about 124 million tweets and then fine-tuned on tweets labelled by humans. Compared with VADER:

| | VADER | Twitter-RoBERTa (transformer) |
|---|---|---|
| How it decides | Looks up words, applies rules | Learned patterns over the whole sentence in context |
| Transparent? | Yes: every number can be traced | No: hundreds of millions of parameters |
| Needs | Nothing (1 MB, instant) | A 500 MB download; slower; a GPU helps for large data |
| Context, negation, slang | Rules for a few patterns | Usually better, because it learned from real tweets |
| What the feeling is *about* | Cannot tell | Still not asked: it rates the whole tweet, not the sentiment toward Facebook |
| Other languages | English only | English model; multilingual versions exist |
| Sarcasm | No | Sometimes |

<span style="color: blue;">Neither model answers our exact question (sentiment *toward Facebook*). That requires **target-based** (aspect-based) sentiment, or an LLM given clear instructions, as in the corporate-venturing notebook.

The cell below is **switched off** by default. It needs `pip install transformers torch` and an internet connection the first time (about 500 MB). If a file `data/transformer_scores.csv` exists, it loads the saved results instead.

<span style="color: blue;">**For the instructor:** run it once with `USE_TRANSFORMER = True` on your own computer. That saves `data/transformer_scores.csv`. Set the switch back to `False`, and from then on the notebook shows the saved RoBERTa results in class without any download.

In [ ]:
USE_TRANSFORMER = False      # 🔧 set to True to download and run the model (not needed in class)
CACHE = DATA / "transformer_scores.csv"

def prep(text):
    # The model's authors replace user names and links with placeholders
    return " ".join("@user" if w.startswith("@") and len(w) > 1 else "http" if w.startswith("http") else w
                    for w in text.split(" "))

if USE_TRANSFORMER:
    from transformers import pipeline
    clf = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
    out = clf([prep(t) for t in excerpts.text], truncation=True)
    trf = pd.DataFrame({"excerpt_id": excerpts.excerpt_id,
                        "transformer_label": [o["label"].lower() for o in out],
                        "transformer_confidence": [round(o["score"], 3) for o in out]})
    trf.to_csv(CACHE, index=False)
elif CACHE.exists():
    trf = pd.read_csv(CACHE)
else:
    trf = None
    print("Transformer comparison is switched off and no saved results were found. That's fine for class.")

if trf is not None:
    cmp = key[["suggested_label", "vader_label"]].join(trf.set_index("excerpt_id"))
    print(f"Agreement with the key: VADER {(cmp.vader_label == cmp.suggested_label).mean():.0%}, "
          f"transformer {(cmp.transformer_label == cmp.suggested_label).mean():.0%}")
    display(cmp)

<span style="color: blue;">**Go deeper (if you ran it):** Which traps does the transformer handle better? H1 and H3 are good tests. Which does it still miss? Look at H5 (spam) and E9 (a magazine name). On 22 hand-picked excerpts, a difference of one or two tweets is not evidence that one model is better.

## <span style="color: green;"> Wrap-up: questions to take away

1. **Target.** "Tweets that mention Facebook" and "opinions about Facebook" are different populations. How would you design data collection so that most of your sample is about the thing you care about?
2. **Validity.** VADER measures how many positive and negative *words* a tweet contains. When is that a good proxy for customer satisfaction, and when is it not?
3. **Cleaning is analysis.** Duplicates, bots, spam and language changed the share of positive tweets. Which choices would you report to an investor, and why?
4. **Method choice.** When would you still pick VADER over a transformer or an LLM? Think about cost, speed, transparency, replicability, and how much text you have.
5. **Checking the machine.** What is the smallest human-coded sample you would accept before trusting an automated sentiment score in a business decision?

---
*<span style="color: green;"> Data: 200 tweets mentioning Facebook, 1 October 2015, with VADER 3.3.2 scores (Hutto, C.J. & Gilbert, E. (2014). VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text. ICWSM). Excerpts, cleaning flags and suggested labels © 2026 Sheryl Winston Smith. For teaching use.*